<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/ContextManagement(ContextEditingMiddleware).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 10.7 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import ContextEditingMiddleware, ModelRequest, ModelResponse, wrap_model_call
from langchain.agents.middleware.context_editing import ClearToolUsesEdit
from langchain_core.messages import AIMessage,HumanMessage
from langchain.tools import tool
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from pydantic import SecretStr
from typing import Annotated, Callable, List

api_key = userdata.get('OPENAI_API_KEY')

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

In [3]:
@tool
def fetch_logs(service: str) -> str:
    """
    Fetch raw application logs for a service over the last N minutes.
    """
    line ="[2026-04-20T10:15:{s:02d}Z] {service} WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-{s:04d}"
    return "\n".join(line.format(s=i % 60, service = service) for i in range(10))

използваме wrap_model_call middleware да вникнем вътре в самия работен цикъл на агента за да видим каква информация достига до модела.

Summarization middleware пренаписваше историята, а този маскира временно , маскира в рамките на конкретното обръщение


In [11]:
@wrap_model_call
def log_model_input(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]):
    print(f"\n--- model sees {len(request.messages)} messages ---")
    for m in request.messages:
        preview = str(getattr(m, "content", ""))[:60].replace("\n", " ")
        extra = f"tool_calls={len(m.tool_calls)}" if hasattr(m, "tool_calls") else ""
        output = [f"{m.type:12s}", preview, extra]

        print(" | ".join(x for x in output if x != ""))

    return handler(request)

ContextEditingMiddleware когато се натрупат много tool_calls просто можем да махнем част от тях. Trigger: работи спрямо и единствено само на база на tokens,
keep - колко от последните tool calls трябва да бъдат запазени.

In [8]:
model = ChatOpenAI(model = "gpt-5-nano", api_key = api_key, reasoning_effort="low")

agent = create_agent(
    model = model,
    tools = [fetch_logs],
    system_prompt=f"You are an on-call SRE assistant. For any incident question, first call `{fetch_logs.name}` for the relevant service, then summarize root-cause signals in plain language. Be concise.",
    checkpointer = InMemorySaver(),
    middleware = [
        ContextEditingMiddleware(
            edits = [
                ClearToolUsesEdit(trigger= 200, keep= 1, clear_tool_inputs = True)
            ]
        ),
        log_model_input
    ]
)

interact = agent | RunnableLambda(lambda res: print_conversation(res['messages']))

In [12]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Fetch logs for the 'application_api' service.")]
    },
    config = {
        "configurable": {
            "thread_id": "thread_2"
        },
        "recursion_limit": 100
    }
)


--- model sees 1 messages ---
human       

--- model sees 3 messages ---
human       
ai           | tool_calls=1
tool        
================================ Human Message =================================

Fetch logs for the 'application_api' service.
================================== Ai Message ==================================
Tool Calls:
  fetch_logs (call_jEPonC1i5ZrM8cNS8OJsRr3i)
 Call ID: call_jEPonC1i5ZrM8cNS8OJsRr3i
  Args:
    service: application_api
================================= Tool Message =================================
Name: fetch_logs

[2026-04-20T10:15:00Z] application_api WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-0000
[2026-04-20T10:15:01Z] application_api WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-0001
[2026-04-20T10:15:02Z] application_api WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-0002
[2026-04-20T10:15:03Z] application_api WARN db pool exhasted retries=3 l

In [13]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Let's explore the 'administration_api' service now.")]
    },
    config = {
        "configurable": {
            "thread_id": "thread_2"
        },
        "recursion_limit": 100
    }
)


--- model sees 5 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       

--- model sees 7 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=1
tool        
================================ Human Message =================================

Fetch logs for the 'application_api' service.
================================== Ai Message ==================================
Tool Calls:
  fetch_logs (call_jEPonC1i5ZrM8cNS8OJsRr3i)
 Call ID: call_jEPonC1i5ZrM8cNS8OJsRr3i
  Args:
    service: application_api
================================= Tool Message =================================
Name: fetch_logs

[2026-04-20T10:15:00Z] application_api WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-0000
[2026-04-20T10:15:01Z] application_api WARN db pool exhasted retries=3 latency_ms=842 trace_id = abc123 user_id=U-0001
[2026-04-20T10:15:02Z] appl

In [14]:
interact.invoke(
    input={
        "messages": [HumanMessage("I need the logs for two more services - 'auth_api' and 'payment_gateway'.")]
    },
    config={
        "configurable": {
            "thread_id": "thread_2"
        }
    }
)


--- model sees 9 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       

--- model sees 12 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=2
tool        
tool        

--- model sees 14 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=2
tool        
tool        
ai           | tool_calls=1
tool        

--- model sees 16 messages ---
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | tool_calls=1
tool        
ai           | tool_calls=0
human       
ai           | t

GraphRecursionError: Recursion limit of 25 reached without hitting a stop condition. You can increase the limit by setting the `recursion_limit` config key.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langgraph/errors/GRAPH_RECURSION_LIMIT